# 10 — Statistical Analysis (single source of statistical inference)

All daily/instance-level inference for the manuscript is produced HERE
and only here, from the corrected de-identified raw outcomes loaded in
notebooks 06/08. Independent unit = n=51 held-out instances (NOT
block-level pseudo-replicated observations -- see manuscript Section 6.6
for the full rationale).

Sign convention (explicit, never direction-blind): for a lateness-type
metric, comparing `a` vs `b`, **r > 0 means `a` has greater/worse
lateness than `b`**.


In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
assert 'REPO_ROOT' in dir(), "Run notebook 00 first."
sys.path.insert(0, REPO_ROOT)
from src.statistics import friedman_omnibus, paired_wilcoxon_report, bootstrap_all
from src.metrics import rank_biserial_signed


## Load corrected de-identified raw outcomes (NR/FR/RG/SA + Liu-ALNS)

In [ ]:
EXP_DIR = os.path.join(REPO_ROOT, "data_deidentified", "experiment_outputs")
main_df = pd.read_csv(os.path.join(EXP_DIR, "nr_fr_rg_sa_block_level.csv"))
liu_df = pd.read_csv(os.path.join(EXP_DIR, "liu_alns_block_level.csv"))
print(f"NR/FR/RG/SA: {len(main_df)} rows")
print(f"Liu-ALNS: {len(liu_df)} rows")


## Aggregate to daily/instance level: trigger x shock x seed within each instance (n=51)

In [ ]:
other_daily = main_df.groupby(['delivery_date','method'])['total_lateness_min'].mean().unstack()[['NR','FR','RG','SA']]
liu_daily = liu_df.groupby('delivery_date')['total_lateness_min'].mean().rename('Liu-ALNS')
daily = other_daily.join(liu_daily)
n = len(daily)
print(f"Independent unit: n={n} held-out instances (expected 51 in FULL mode)")
if RUN_MODE == "full":
    assert n == 51, f"Expected 51 instances, got {n}"
print(daily.head())

results_dir = os.path.join(REPO_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)
daily.to_csv(os.path.join(results_dir, "daily_instance_5method_lateness.csv"))


## Friedman omnibus (5 methods)

In [ ]:
friedman_result = friedman_omnibus(daily, ['NR','FR','RG','SA','Liu-ALNS'])
print(json.dumps(friedman_result, indent=2, default=float))


## Planned paired Wilcoxon comparisons

Includes the comparator-focused tests (SA vs Liu-ALNS, Liu-ALNS vs FR,
Liu-ALNS vs RG) AND the internal NR/FR/RG/SA comparisons the manuscript
also needs (SA vs NR/FR/RG, FR vs RG), all under one Holm family-wise
correction.


In [ ]:
comparisons = [
    ('SA','NR'), ('SA','FR'), ('SA','RG'), ('FR','RG'),        # internal
    ('SA','Liu-ALNS'), ('Liu-ALNS','FR'), ('Liu-ALNS','RG'),   # comparator-focused
]
wilcoxon_rows = paired_wilcoxon_report(daily, comparisons)
wilcoxon_df = pd.DataFrame(wilcoxon_rows)
print(wilcoxon_df.to_string(index=False))
print("\nSign convention: r > 0 means the FIRST named method has GREATER/WORSE lateness than the second.")
wilcoxon_df.to_csv(os.path.join(results_dir, "wilcoxon_holm_results.csv"), index=False)


## 10,000 instance-cluster bootstrap 95% CIs

In [ ]:
N_BOOT = 200 if RUN_MODE == "quick" else 10000
boot_results = bootstrap_all(daily, comparisons, n_boot=N_BOOT, seed=42)
boot_df = pd.DataFrame(boot_results)
print(boot_df.to_string(index=False))
boot_df.to_csv(os.path.join(results_dir, "bootstrap_ci_results.csv"), index=False)
if RUN_MODE == "quick":
    print(f"\n[QUICK MODE] n_boot={N_BOOT} (reduced from 10,000) -- CIs are wider/noisier than the")
    print("              manuscript's reported values; this is a smoke test, not a research result.")


## Test metadata (saved for full audit trail)

In [ ]:
metadata = dict(
    n_instances=n, run_mode=RUN_MODE, n_boot=N_BOOT, bootstrap_seed=42,
    sign_convention="r > 0 means the first named method has greater/worse lateness",
    correction_method="Holm step-down (family-wise, across all listed comparisons together)",
    independent_unit="held-out instance (date), NOT block-level pseudo-replicated observation",
)
with open(os.path.join(results_dir, "statistical_test_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))


## Expected outputs / integrity checks

In [ ]:
checks = {
    "daily_aggregation_ran": n > 0,
    "friedman_ran": friedman_result['p'] is not None,
    "wilcoxon_all_comparisons_ran": len(wilcoxon_df) == len(comparisons),
    "holm_applied": 'p_holm' in wilcoxon_df.columns,
    "bootstrap_ran": len(boot_df) == len(comparisons),
    "sign_convention_documented": True,
}
if RUN_MODE == "full":
    checks["exact_n_51"] = n == 51
    checks["exact_n_boot_10000"] = N_BOOT == 10000

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_10_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 10 STATUS: {NOTEBOOK_10_STATUS}")
assert NOTEBOOK_10_STATUS == "PASS"
